# DSA PART 3 - GRAPHS & SORTING

# Graphs

## Introduction
Your DSA Toolkit already has `LinkedList`, `Stack`, `Queue`, `BST`, and `HashMap` (Lessons 41-42). A **graph** is the most general structure of all of them: a set of nodes (**vertices**) connected by **edges**, with no restriction on how many neighbours a node can have or in what order they connect. Social networks, road maps, dependency trees, and web links are all graphs.

Today you add a `Graph` class to the toolkit, built on an **adjacency list** representation, and give it two classic traversal methods: **Breadth-First Search (BFS)** and **Depth-First Search (DFS)** - the graph equivalents of the traversals you already wrote for `BST` in Lesson 42.

| Structure (so far) | Shape | Typical use |
|---|---|---|
| `LinkedList` | linear chain | ordered sequence, cheap insert/delete |
| `Stack` / `Queue` | linear, restricted access | LIFO / FIFO processing |
| `BST` | hierarchical, ordered | fast ordered search |
| `HashMap` | key -> bucket | O(1) average lookup |
| `Graph` | arbitrary connections | networks, routes, dependencies |


## Learning Objectives
By the end of this lesson, you should be able to:
- Represent a graph using an adjacency list in Python
- Explain the trade-off between an adjacency list and an adjacency matrix
- Implement Breadth-First Search (BFS) using a queue
- Implement Depth-First Search (DFS) using recursion and using an explicit stack
- Implement bubble sort, insertion sort, merge sort, and quicksort from scratch
- State the Big-O time and space complexity of every structure and sort covered across DSA Parts 1-3
- Explain, in one sentence, what Timsort is and why Python's `sorted()` uses it


## Adjacency List Representation
A graph's **adjacency list** maps each vertex to the list of vertices it connects to. In Python this is naturally a `dict` of `key -> list` (or `set`, if duplicate edges should be rejected) - the same `HashMap` concept from Lesson 42, just applied to relationships instead of records.

### Syntax
```python
class Graph:
    def __init__(self, directed: bool = False):
        self._adjacency: dict[str, list[str]] = {}
        self._directed = directed

    def add_vertex(self, vertex: str) -> None:
        self._adjacency.setdefault(vertex, [])

    def add_edge(self, source: str, destination: str) -> None:
        self.add_vertex(source)
        self.add_vertex(destination)
        self._adjacency[source].append(destination)
        if not self._directed:
            self._adjacency[destination].append(source)
```

### Illustration
```python
g = Graph()
g.add_edge("A", "B")
g.add_edge("A", "C")
g.add_edge("B", "D")
print(g._adjacency)
# -> {'A': ['B', 'C'], 'B': ['A', 'D'], 'C': ['A'], 'D': ['B']}
```

### Explanation
`add_vertex` uses `.setdefault()` (Part II, Day 3) so re-adding an existing vertex is harmless. `add_edge` records the connection in both directions unless the graph is `directed`, matching a plain (undirected) friendship-style network. The whole structure is just a `dict` whose values are `list`s - no new container type needed, only a new way of arranging familiar ones.

## Breadth-First Search (BFS)
BFS explores a graph **level by level**: it visits every neighbour of the starting vertex before moving further out. This "closest first" behaviour is exactly what a `Queue` (FIFO, Lesson 41) is built for.

### Syntax
```python
from collections import deque

def bfs(self, start: str) -> list[str]:
    visited = {start}
    order = []
    queue = deque([start])
    while queue:
        vertex = queue.popleft()
        order.append(vertex)
        for neighbour in self._adjacency[vertex]:
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append(neighbour)
    return order
```

### Illustration
```python
g.bfs("A")
# -> ['A', 'B', 'C', 'D']
```

### Explanation
`visited` (a `set`, Part II Day 3) prevents revisiting a vertex and looping forever on a cyclic graph. Every vertex is enqueued once, so BFS runs in O(V + E) - visiting every vertex and every edge exactly once. `deque` (Part II, Day 9) is used instead of a plain list because popping from the front of a list is O(n), while `deque.popleft()` is O(1).

### Working Example 1
Add `add_vertex`, `add_edge`, and `bfs` to a new `Graph` class. Read the road network's edges from the
provided `Data/road_network.csv` (columns `from,to` -- large enough that the visiting order is not
obvious just by looking at the file) using `csv.reader` (Lesson 15) to build the graph instead of typing
the edges by hand, then print the BFS visiting order starting from `"Town A"`. As with any provided data
file (Lesson 17's "A Note on Real-World Data"), a few rows in this file are missing the `from` or `to`
town, or are blank -- a row you can't turn into a valid edge should simply be skipped while building the
graph, not allowed to crash `add_edge()`.

```markdown
Sample Output
Edges loaded from Data/road_network.csv: N edges across M towns (K rows skipped as unusable)

BFS from Town A:
['Town A', 'Town B', ...]
```

*PyTech Hub*

In [ ]:
# working example 1 code


## Depth-First Search (DFS)
DFS explores **as far as possible along one branch** before backtracking - the opposite instinct to BFS. It can be written two ways: **recursively** (letting Python's own call stack, Part I Day 10, do the backtracking for you) or **iteratively** with an explicit `Stack` (Lesson 41), since "go deep, then come back" is exactly LIFO behaviour.

### Syntax
```python
def dfs_recursive(self, start: str, visited: set | None = None) -> list[str]:
    if visited is None:
        visited = set()
    visited.add(start)
    order = [start]
    for neighbour in self._adjacency[start]:
        if neighbour not in visited:
            order.extend(self.dfs_recursive(neighbour, visited))
    return order

def dfs_iterative(self, start: str) -> list[str]:
    visited = set()
    order = []
    stack = [start]
    while stack:
        vertex = stack.pop()
        if vertex not in visited:
            visited.add(vertex)
            order.append(vertex)
            stack.extend(reversed(self._adjacency[vertex]))
    return order
```

### Illustration
```python
g.dfs_recursive("A")   # -> ['A', 'B', 'D', 'C']
g.dfs_iterative("A")   # -> ['A', 'B', 'D', 'C']
```

### Explanation
`dfs_recursive` mirrors the recursive `BST` traversals from Lesson 42: base case is "no unvisited neighbours left", recursive case is "descend into the next unvisited neighbour". `dfs_iterative` uses a plain Python `list` as a stack (`.append()`/`.pop()`, Part II Day 2) with `reversed()` so the traversal order matches the recursive version. Both run in O(V + E), same as BFS - the difference is *order*, not cost.

### Working Example 2
Add both `dfs_recursive` and `dfs_iterative` to `Graph`. Run both on the road network from Working Example 1 starting at `"Town A"` and confirm they produce the same order.

```markdown
Sample Output
DFS (recursive) from Town A: ['Town A', 'Town B', ...]
DFS (iterative) from Town A: ['Town A', 'Town B', ...]
```

*PyTech Hub*

In [ ]:
# working example 2 code


# Sorting

## Introduction
You have used `sorted()` and `.sort()` since Part I - now you build the algorithms behind them. Sorting is the classic Big-O teaching ground: the same task (arranging a list) done four different ways, with dramatically different time complexity. You will implement two simple O(n^2) warm-up sorts, then two efficient O(n log n) sorts, and finish with the master Big-O comparison table across everything built in DSA Parts 1-3.

## Bubble Sort & Insertion Sort (warm-ups)
These two sorts are taught together because they share the same idea - repeatedly compare neighbouring or nearby elements and swap - and both are O(n^2), making them useful for *understanding* sorting before tackling the faster divide-and-conquer sorts below.

### Syntax
```python
def bubble_sort(items: list) -> list:
    items = items.copy()
    n = len(items)
    for i in range(n):
        for j in range(0, n - i - 1):
            if items[j] > items[j + 1]:
                items[j], items[j + 1] = items[j + 1], items[j]
    return items

def insertion_sort(items: list) -> list:
    items = items.copy()
    for i in range(1, len(items)):
        key = items[i]
        j = i - 1
        while j >= 0 and items[j] > key:
            items[j + 1] = items[j]
            j -= 1
        items[j + 1] = key
    return items
```

### Illustration
```python
bubble_sort([5, 2, 4, 1])      # -> [1, 2, 4, 5]
insertion_sort([5, 2, 4, 1])   # -> [1, 2, 4, 5]
```

### Explanation
Bubble sort repeatedly "bubbles" the largest unsorted element to the end via adjacent swaps - `n` passes of up to `n` comparisons gives O(n^2). Insertion sort builds the sorted portion one element at a time, shifting larger elements right to make room - same O(n^2) worst case, but noticeably faster in practice on nearly-sorted data, since the inner `while` exits early.

### Working Example 3
Write `bubble_sort()` and `insertion_sort()`. Sort `[64, 34, 25, 12, 22, 11, 90]` with each and print both results plus how many swaps bubble sort performed.

```markdown
Sample Output
Bubble sort result: [11, 12, 22, 25, 34, 64, 90]
Swaps: 15
Insertion sort result: [11, 12, 22, 25, 34, 64, 90]
```

*PyTech Hub*

In [ ]:
# working example 3 code


## Merge Sort
Merge sort is a **divide-and-conquer** algorithm: split the list in half recursively until each piece has one element (trivially sorted), then **merge** sorted halves back together. This is the same recursive "shrink the problem" thinking from Part I, Day 10 recursion.

### Syntax
```python
def merge_sort(items: list) -> list:
    if len(items) <= 1:
        return items
    mid = len(items) // 2
    left = merge_sort(items[:mid])
    right = merge_sort(items[mid:])
    return _merge(left, right)

def _merge(left: list, right: list) -> list:
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i]); i += 1
        else:
            result.append(right[j]); j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result
```

### Illustration
```python
merge_sort([5, 2, 4, 1, 3])
# -> [1, 2, 3, 4, 5]
```

### Explanation
The base case is a list of length 0 or 1 (already sorted). Each recursive call halves the input, giving `log n` levels of splitting; merging two halves costs O(n) per level, for a total of O(n log n) - asymptotically better than the O(n^2) warm-up sorts, at the cost of O(n) extra space for the merged lists.

### Working Example 4
Write `merge_sort()` (and a helper `_merge()`). Sort `[38, 27, 43, 3, 9, 82, 10]` and print the result.

```markdown
Sample Output
Merge sort result: [3, 9, 10, 27, 38, 43, 82]
```

*PyTech Hub*

In [ ]:
# working example 4 code


## Quicksort
Quicksort is also divide-and-conquer, but instead of splitting evenly it picks a **pivot**, partitions everything smaller to its left and everything larger to its right, then recursively sorts each side. Average case O(n log n); worst case (already-sorted input with a poor pivot choice) degrades to O(n^2).

### Syntax
```python
def quicksort(items: list) -> list:
    if len(items) <= 1:
        return items
    pivot = items[len(items) // 2]
    left = [x for x in items if x < pivot]
    middle = [x for x in items if x == pivot]
    right = [x for x in items if x > pivot]
    return quicksort(left) + middle + quicksort(right)
```

### Illustration
```python
quicksort([5, 2, 4, 1, 3])
# -> [1, 2, 3, 4, 5]
```

### Explanation
The three list comprehensions (Part II, Day 7) partition the input around `pivot` in one O(n) pass; `quicksort` then recurses only on `left` and `right`. Unlike merge sort, quicksort has no separate merge step - concatenation does the job - but its worst case is worse because a badly chosen pivot can produce lopsided partitions.

### Working Example 5
Write `quicksort()`. Sort `[33, 10, 68, 21, 90, 5]` and print the result, then confirm it matches `sorted()`'s output for the same list.

```markdown
Sample Output
Quicksort result: [5, 10, 21, 33, 68, 90]
Matches sorted(): True
```

*PyTech Hub*

In [ ]:
# working example 5 code


## The Big-O Comparison Table
Python's own `sorted()` and `list.sort()` do not use any of the four sorts above - they use **Timsort**, a hybrid of merge sort and insertion sort tuned for real-world data (which is often partially sorted already). You do not need to implement Timsort; just know the name and that it is O(n log n) worst case with excellent best-case performance on nearly-sorted input.

### Structures & Sorts - Time Complexity Summary
| Structure / Sort | Best | Average | Worst | Space |
|---|---|---|---|---|
| `list` index access | O(1) | O(1) | O(1) | O(n) |
| `LinkedList` search | O(1) | O(n) | O(n) | O(n) |
| `Stack` / `Queue` push-pop | O(1) | O(1) | O(1) | O(n) |
| `BST` search | O(log n) | O(log n) | O(n) | O(n) |
| `HashMap` (`dict`) lookup | O(1) | O(1) | O(n) | O(n) |
| `Graph` BFS/DFS | O(V+E) | O(V+E) | O(V+E) | O(V) |
| Bubble sort | O(n) | O(n^2) | O(n^2) | O(1) |
| Insertion sort | O(n) | O(n^2) | O(n^2) | O(1) |
| Merge sort | O(n log n) | O(n log n) | O(n log n) | O(n) |
| Quicksort | O(n log n) | O(n log n) | O(n^2) | O(log n) |
| Timsort (`sorted()`) | O(n) | O(n log n) | O(n log n) | O(n) |


### Working Example 6
Benchmark `bubble_sort`, `insertion_sort`, `merge_sort`, `quicksort`, and built-in `sorted()` on the same random list of 2,000 integers using `time.perf_counter()`. Print a formatted comparison table of the timings, sorted fastest to slowest.

```markdown
Sample Output
Sort            Time (s)
--------------------------
sorted()          0.0011
quicksort         0.0087
merge_sort        0.0102
insertion_sort    0.8341
bubble_sort       1.2765
```

*PyTech Hub*

In [ ]:
# working example 6 code


## Tutor Demonstration

### CHALLENGE 1
Extend the `Graph` class with a `shortest_path_length(start, end)` method that uses BFS to find the number of edges on the shortest path between two vertices (unweighted graph), returning `-1` if no path exists.

**Illustration**
```markdown
Graph: A-B, B-C, C-D, A-D
shortest_path_length("A", "C") -> 2
shortest_path_length("A", "Z") -> -1
```

**Explanation**
BFS naturally discovers the shortest path first in an unweighted graph, because it explores level by level - track each vertex's distance from `start` as it is first visited.

**Requirements**
- Reuse the `bfs`-style queue/visited pattern
- Must return an `int`, never raise an exception for a missing vertex/path

*PyTech Hub*

In [ ]:
# challenge 1 code


### CHALLENGE 2
Write a `is_sorted_check(sort_function, data)` helper that runs any of today's sort functions on `data`, verifies the output is actually sorted and contains the same elements as the input (no data loss), and returns `True`/`False`. Use it to test all four sorts on three different lists: already sorted, reverse sorted, and random.

**Illustration**
```markdown
bubble_sort on reverse-sorted list: PASSED
quicksort on random list: PASSED
```

**Explanation**
Compare the function's output against Python's own `sorted(data)` for correctness, and compare `sorted(output) == sorted(data)` (or use `Counter`, Part II Day 3) to confirm no elements were dropped or duplicated.

**Requirements**
- Must test all four custom sort functions
- Must test at least 3 differently-shaped input lists per sort

*PyTech Hub*

In [ ]:
# challenge 2 code


## Summary
At the end of the lesson, these are the concepts I want you to:

**Memorize**
1. Adjacency list pattern: `dict[vertex] -> list[neighbours]`
2. BFS uses a queue (`collections.deque`); DFS uses recursion or an explicit stack
3. Both BFS and DFS run in O(V + E)
4. Bubble sort / insertion sort: O(n^2); merge sort / quicksort: O(n log n) average
5. Merge sort core shape:
```python
def merge_sort(items):
    if len(items) <= 1:
        return items
    mid = len(items) // 2
    return _merge(merge_sort(items[:mid]), merge_sort(items[mid:]))
```
6. Python's `sorted()` and `.sort()` use Timsort

**Understand**
1. Why adjacency lists are usually preferred over adjacency matrices for sparse graphs
2. Why quicksort's worst case is worse than merge sort's despite similar average performance
3. Why insertion sort outperforms bubble sort in practice despite the same Big-O class
4. How today's Big-O comparison table lets you choose the right structure/sort for a given problem


PYTHECH HUB

Contact Tutor for Assistance via the following:
* WhatsApp: https://wa.me/233209130538/
* Email: pytech.hub@gmail.com


## MINI PROJECT 43
Project code: **mp43**

Difficulty Rating: **8/10**

### Problem
Extend your DSA Toolkit (Lessons 41-42) with a `Graph` class (adjacency list, `add_edge`, `bfs`, `dfs_recursive`, `dfs_iterative`) and a standalone `sorting` module containing `bubble_sort`, `insertion_sort`, `merge_sort`, and `quicksort`. Build a demo dataset of at least 1,000 random integers and a small graph (at least 6 vertices), then benchmark all four sorts against `sorted()` and print the final Big-O comparison table (from today's lesson) alongside your measured timings.

### Illustration
```markdown
=== Graph Demo ===
BFS from A: ['A', 'B', 'C', 'D', 'E', 'F']
DFS from A: ['A', 'B', 'D', 'E', 'C', 'F']

=== Sorting Benchmark (n=1000) ===
Sort            Time (s)
--------------------------
sorted()          0.0006
quicksort         0.0041
merge_sort        0.0049
insertion_sort    0.1872
bubble_sort       0.2931

=== Big-O Comparison Table ===
(structure/sort summary printed from Lesson 43)
```

### Explanation
Every method must carry a Big-O-annotated docstring (the habit started in Lesson 41). The benchmark should time each sort with `time.perf_counter()` on an identical copy of the dataset (use `.copy()` so earlier sorts don't leave the list already sorted for later ones), then print results ordered fastest to slowest.

*PyTech Hub*

### Submission
See the steps below to submit your mini-project:

0. Take note of the project code and your ID (ask for help if you don't know them).
1. Create a python file (FileName.py)
2. For file should be named according to the following steps
    - Project code: **mp43**
    - Participants ID: **A45**
    - FileName: **mp43_A45.py**
3. Submit your project via google forms: [https://forms.gle/Lm6J2cTXTQJ3GLrF6](https://forms.gle/Lm6J2cTXTQJ3GLrF6)


In [ ]:
# Mini Project 43 Code


## Appendix
Extra detail and related notes to glance through - not required to master the core ideas above, but useful for going deeper.

### Adjacency Matrix (the alternative graph representation)

#### Explanation
An **adjacency matrix** is a V x V grid of 0s and 1s (or weights), where `matrix[i][j]` records whether an edge exists between vertex `i` and vertex `j`. It trades memory for speed on one specific question: "are these two vertices directly connected?"

#### Syntax
```python
# matrix[i][j] = 1 if an edge exists between vertex i and vertex j
matrix = [[0] * n for _ in range(n)]
```

#### Illustration
```python
# 4 vertices: A=0, B=1, C=2, D=3; edges A-B, B-C
matrix = [
    [0, 1, 0, 0],
    [1, 0, 1, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 0],
]
```
```markdown
matrix[0][1] -> 1  (A-B connected)
matrix[0][2] -> 0  (A-C not connected)
```

#### Explanation
Checking `matrix[i][j]` is O(1), faster than scanning an adjacency list's neighbour list. But the matrix uses O(V^2) space regardless of how many edges actually exist, which wastes huge amounts of memory on **sparse** graphs (few edges relative to vertices) - most real-world graphs (social networks, road maps) are sparse, which is why adjacency lists (O(V + E) space) are the more common default.

| | Adjacency List | Adjacency Matrix |
|---|---|---|
| Space | O(V + E) | O(V^2) |
| Check edge exists | O(neighbours) | O(1) |
| Best for | sparse graphs | dense graphs |


### `sorted()`'s Timsort, briefly

#### Explanation
Timsort (used internally by both `sorted()` and `list.sort()`) breaks the input into small chunks, sorts each chunk with insertion sort, then merges the chunks with merge sort's merge step. This hybrid exploits the fact that real-world data is often already partially ordered, giving it a best case of O(n) - something none of today's four custom sorts achieve. You never need to implement Timsort yourself; knowing it exists (and why Python trusts `sorted()` over a hand-rolled sort in production code) is enough.